# ChromaDB Storage Layer
**IST 488 Final Project — Milestone (Apr 19, 2026)**
**Owner:** Toby

> ⚠️ **Backup draft** prepared by Ryan as a starting point. Reranking strategy and embedding model are open — customize as you see fit.

**Pipeline position:**
`Lauren's discovery → Ryan's scraper → Leytisha's enrichment → [THIS LAYER: ChromaDB store + retrieval]`

**What this notebook does:**
1. Sets up a **persistent ChromaDB** collection (writes to `./chroma_store`).
2. Loads enriched restaurant records from `restaurants_enriched.json`.
3. Builds a meaningful embedding text per restaurant (name + city + cuisine + menu items).
4. Provides `chroma_search(query, k)` — the function Leytisha's query agent will call.
5. Includes an **MVP rerank step** that boosts results by Google rating and city match.
6. Persistent on-disk storage = **long-term memory** (satisfies instructor req #1, long-term half) + **RAG with reranking** (satisfies req #2).

**Embedding model:** Chroma's default `all-MiniLM-L6-v2` (free, fast, runs locally). Easy to swap for OpenAI embeddings later if quality demands it.


## 1. Install dependencies

In [ ]:
!pip install -q chromadb sentence-transformers


## 2. Imports & config

In [ ]:
import json
from pathlib import Path
from typing import Optional

import chromadb
from chromadb.config import Settings

CHROMA_DIR = Path("./chroma_store")
COLLECTION_NAME = "restaurants_v1"

# Persistent client = data survives Colab session restart (long-term memory)
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))


## 3. Bundled fallback data (if `restaurants_enriched.json` not present)

In [ ]:
SAMPLE_RESTAURANTS = [
    # --- Syracuse ---
    {"place_id": "syr_001", "name": "Dinosaur Bar-B-Que", "address": "246 W Willow St, Syracuse, NY 13202",
     "city": "Syracuse", "rating": 4.5, "user_rating_count": 6800, "price_level": "MODERATE",
     "website_url": "https://www.dinosaurbarbque.com/", "types": ["restaurant", "barbecue_restaurant"]},
    {"place_id": "syr_002", "name": "Pastabilities", "address": "311 S Franklin St, Syracuse, NY 13202",
     "city": "Syracuse", "rating": 4.4, "user_rating_count": 2100, "price_level": "MODERATE",
     "website_url": "https://pastabilities.com/", "types": ["restaurant", "italian_restaurant"]},
    {"place_id": "syr_003", "name": "Apizza Regionale", "address": "260 W Genesee St, Syracuse, NY 13202",
     "city": "Syracuse", "rating": 4.4, "user_rating_count": 950, "price_level": "MODERATE",
     "website_url": "https://apizzaregionale.com/", "types": ["restaurant", "pizza_restaurant"]},
    {"place_id": "syr_004", "name": "Phoebe's Restaurant", "address": "900 E Genesee St, Syracuse, NY 13210",
     "city": "Syracuse", "rating": 4.3, "user_rating_count": 780, "price_level": "MODERATE",
     "website_url": "https://phoebesrestaurant.com/", "types": ["restaurant", "american_restaurant"]},
    {"place_id": "syr_005", "name": "Kitty Hoynes Irish Pub", "address": "301 W Fayette St, Syracuse, NY 13202",
     "city": "Syracuse", "rating": 4.4, "user_rating_count": 1500, "price_level": "MODERATE",
     "website_url": "https://kittyhoynes.com/", "types": ["restaurant", "irish_pub"]},
    {"place_id": "syr_006", "name": "Otro Cinco", "address": "206 S Warren St, Syracuse, NY 13202",
     "city": "Syracuse", "rating": 4.3, "user_rating_count": 620, "price_level": "MODERATE",
     "website_url": "https://otrocincorestaurant.com/", "types": ["restaurant", "mexican_restaurant"]},
    {"place_id": "syr_007", "name": "The Fish Friar", "address": "247 W Fayette St, Syracuse, NY 13202",
     "city": "Syracuse", "rating": 4.5, "user_rating_count": 540, "price_level": "EXPENSIVE",
     "website_url": "https://www.fishfriar.com/", "types": ["restaurant", "seafood_restaurant"]},
    # --- Rochester ---
    {"place_id": "roc_001", "name": "Nick Tahou Hots", "address": "320 W Main St, Rochester, NY 14608",
     "city": "Rochester", "rating": 4.2, "user_rating_count": 2300, "price_level": "INEXPENSIVE",
     "website_url": "https://nicktahous.com/", "types": ["restaurant", "american_restaurant"]},
    {"place_id": "roc_002", "name": "Dinosaur Bar-B-Que Rochester", "address": "99 Court St, Rochester, NY 14604",
     "city": "Rochester", "rating": 4.5, "user_rating_count": 4900, "price_level": "MODERATE",
     "website_url": "https://www.dinosaurbarbque.com/", "types": ["restaurant", "barbecue_restaurant"]},
    {"place_id": "roc_003", "name": "Good Luck", "address": "50 Anderson Ave, Rochester, NY 14607",
     "city": "Rochester", "rating": 4.6, "user_rating_count": 1100, "price_level": "EXPENSIVE",
     "website_url": "https://restaurantgoodluck.com/", "types": ["restaurant", "american_restaurant"]},
    {"place_id": "roc_004", "name": "Han Noodle Bar", "address": "687 Monroe Ave, Rochester, NY 14607",
     "city": "Rochester", "rating": 4.5, "user_rating_count": 1800, "price_level": "MODERATE",
     "website_url": "https://hannoodlebar.com/", "types": ["restaurant", "chinese_restaurant"]},
    {"place_id": "roc_005", "name": "Lento", "address": "274 N Goodman St, Rochester, NY 14607",
     "city": "Rochester", "rating": 4.5, "user_rating_count": 700, "price_level": "EXPENSIVE",
     "website_url": "https://lentorestaurant.com/", "types": ["restaurant", "american_restaurant"]},
    {"place_id": "roc_006", "name": "Atlas Eats", "address": "2185 N Clinton Ave, Rochester, NY 14621",
     "city": "Rochester", "rating": 4.6, "user_rating_count": 850, "price_level": "MODERATE",
     "website_url": "https://atlaseats.com/", "types": ["restaurant", "mediterranean_restaurant"]},
    # --- Albany ---
    {"place_id": "alb_001", "name": "Cafe Capriccio", "address": "49 Grand St, Albany, NY 12202",
     "city": "Albany", "rating": 4.6, "user_rating_count": 920, "price_level": "EXPENSIVE",
     "website_url": "https://cafecapriccio.com/", "types": ["restaurant", "italian_restaurant"]},
    {"place_id": "alb_002", "name": "Jack's Oyster House", "address": "42 State St, Albany, NY 12207",
     "city": "Albany", "rating": 4.4, "user_rating_count": 1300, "price_level": "EXPENSIVE",
     "website_url": "https://jacksoysterhouse.com/", "types": ["restaurant", "seafood_restaurant"]},
    {"place_id": "alb_003", "name": "The Hollow Bar + Kitchen", "address": "79 N Pearl St, Albany, NY 12207",
     "city": "Albany", "rating": 4.4, "user_rating_count": 1100, "price_level": "MODERATE",
     "website_url": "https://thehollowalbany.com/", "types": ["restaurant", "american_restaurant"]},
    {"place_id": "alb_004", "name": "El Loco Mexican Cafe", "address": "465 Madison Ave, Albany, NY 12208",
     "city": "Albany", "rating": 4.4, "user_rating_count": 950, "price_level": "MODERATE",
     "website_url": "https://ellococafe.com/", "types": ["restaurant", "mexican_restaurant"]},
    {"place_id": "alb_005", "name": "Yono's Restaurant", "address": "25 Chapel St, Albany, NY 12210",
     "city": "Albany", "rating": 4.6, "user_rating_count": 580, "price_level": "EXPENSIVE",
     "website_url": "https://yonosrestaurant.com/", "types": ["restaurant", "fine_dining"]},
    {"place_id": "alb_006", "name": "Cardona's Market", "address": "340 Delaware Ave, Albany, NY 12209",
     "city": "Albany", "rating": 4.6, "user_rating_count": 720, "price_level": "MODERATE",
     "website_url": "https://cardonasmarket.com/", "types": ["restaurant", "italian_restaurant"]},
    {"place_id": "alb_007", "name": "New World Bistro Bar", "address": "300 Delaware Ave, Albany, NY 12209",
     "city": "Albany", "rating": 4.4, "user_rating_count": 880, "price_level": "MODERATE",
     "website_url": "https://newworldbistrobar.com/", "types": ["restaurant", "american_restaurant"]},
]

# Fake-enrich the mock so this notebook runs end-to-end in isolation
def _mock_enrich(r):
    cuisine_map = {
        "barbecue_restaurant": "BBQ",
        "italian_restaurant": "Italian",
        "pizza_restaurant": "Pizza",
        "american_restaurant": "American",
        "irish_pub": "Irish Pub",
        "mexican_restaurant": "Mexican",
        "seafood_restaurant": "Seafood",
        "chinese_restaurant": "Chinese",
        "mediterranean_restaurant": "Mediterranean",
        "fine_dining": "Fine Dining",
    }
    cuisine = "American"
    for t in r.get("types", []):
        if t in cuisine_map:
            cuisine = cuisine_map[t]
            break
    return {
        **r,
        "cuisine_type": cuisine,
        "menu_items": [],   # real version comes from Leytisha's enrichment
        "price_range": {"INEXPENSIVE": "$", "MODERATE": "$$", "EXPENSIVE": "$$$"}.get(r.get("price_level"), "$$"),
        "menu_available_online": True,
        "scrape_failed": False,
    }

MOCK_ENRICHED = [_mock_enrich(r) for r in SAMPLE_RESTAURANTS]


## 4. Build embedding text per restaurant
Key design choice: what text gets embedded determines what queries match well.

In [ ]:
def build_embedding_text(record: dict) -> str:
    """Compose the searchable text for a restaurant."""
    parts = [
        record.get("name", ""),
        f"in {record.get('city', '')}",
        f"cuisine: {record.get('cuisine_type', '')}",
        f"price: {record.get('price_range', '')}",
    ]
    items = record.get("menu_items") or []
    if items:
        item_names = [
            (it.get("name") if isinstance(it, dict) else str(it))
            for it in items
        ]
        parts.append("menu includes: " + ", ".join(item_names))
    return " | ".join(p for p in parts if p)


## 5. Load records into ChromaDB

In [ ]:
def load_records_to_chroma(records: list[dict], collection_name: str = COLLECTION_NAME):
    """(Re)create collection and bulk-add records."""
    # Delete + recreate for clean reload during dev
    try:
        chroma_client.delete_collection(collection_name)
    except Exception:
        pass

    collection = chroma_client.create_collection(
        name=collection_name,
        metadata={"hnsw:space": "cosine"},
    )

    ids = [r["place_id"] for r in records]
    docs = [build_embedding_text(r) for r in records]
    # Chroma metadata values must be str/int/float/bool — flatten lists to strings.
    metas = [{
        "name": r.get("name", ""),
        "city": r.get("city", ""),
        "address": r.get("address", ""),
        "cuisine_type": r.get("cuisine_type", ""),
        "price_range": r.get("price_range", ""),
        "rating": float(r.get("rating") or 0),
        "website_url": r.get("website_url", ""),
        "menu_items_json": json.dumps(r.get("menu_items", [])),  # rehydrate on read
    } for r in records]

    collection.add(ids=ids, documents=docs, metadatas=metas)
    print(f"Loaded {len(ids)} records into '{collection_name}'")
    return collection


# Try to load Leytisha's enriched output, fall back to mock
enriched_path = Path("restaurants_enriched.json")
if enriched_path.exists():
    records = json.loads(enriched_path.read_text())
    print(f"Using {len(records)} records from {enriched_path}")
else:
    records = MOCK_ENRICHED
    print(f"⚠️  {enriched_path} not found — using {len(records)} mock records")

collection = load_records_to_chroma(records)


## 6. Search + rerank
MVP rerank: combine vector similarity with rating boost and optional city filter.

In [ ]:
def _rerank(results: list[dict], prefer_city: Optional[str] = None) -> list[dict]:
    """Boost score for higher-rated places and city match."""
    for r in results:
        score = r["distance"]                     # lower = better (cosine distance)
        rating = r["metadata"].get("rating", 0)
        score -= 0.05 * (rating - 4.0)            # bonus for >4★
        if prefer_city and r["metadata"].get("city", "").lower() == prefer_city.lower():
            score -= 0.15                         # strong bonus for city match
        r["rerank_score"] = score
    results.sort(key=lambda r: r["rerank_score"])
    return results


def chroma_search(query: str, k: int = 5, prefer_city: Optional[str] = None,
                  collection_name: str = COLLECTION_NAME) -> list[dict]:
    """
    Main retrieval function. This is what Leytisha's query agent calls.

    Returns:
        list of dicts: [{name, city, cuisine_type, price_range, rating,
                         website_url, menu_items, distance, rerank_score}, ...]
    """
    coll = chroma_client.get_collection(collection_name)
    raw = coll.query(query_texts=[query], n_results=k * 2)  # over-fetch for rerank

    results = []
    for i in range(len(raw["ids"][0])):
        meta = raw["metadatas"][0][i]
        results.append({
            "id": raw["ids"][0][i],
            "document": raw["documents"][0][i],
            "metadata": meta,
            "distance": raw["distances"][0][i],
        })

    reranked = _rerank(results, prefer_city=prefer_city)[:k]

    # Flatten + rehydrate menu_items for the consumer
    return [{
        "place_id": r["id"],
        "name": r["metadata"]["name"],
        "city": r["metadata"]["city"],
        "address": r["metadata"]["address"],
        "cuisine_type": r["metadata"]["cuisine_type"],
        "price_range": r["metadata"]["price_range"],
        "rating": r["metadata"]["rating"],
        "website_url": r["metadata"]["website_url"],
        "menu_items": json.loads(r["metadata"].get("menu_items_json", "[]")),
        "_distance": round(r["distance"], 4),
        "_rerank_score": round(r["rerank_score"], 4),
    } for r in reranked]


## 7. Test queries

In [ ]:
test_queries = [
    ("BBQ in Syracuse", "Syracuse"),
    ("authentic Italian dinner", None),
    ("affordable Mexican food", None),
    ("upscale seafood Albany", "Albany"),
]

for q, city in test_queries:
    print(f"\n=== Query: {q!r}  (prefer_city={city}) ===")
    hits = chroma_search(q, k=3, prefer_city=city)
    for h in hits:
        print(f"  • {h['name']} ({h['city']}) — {h['cuisine_type']} — "
              f"{h['rating']}★ — score={h['_rerank_score']}")


## 8. Milestone checklist (Apr 19)

- [x] ChromaDB persistent client set up (long-term memory — req #1 ✓)
- [x] Loads ≥20 records from enriched JSON (or mock fallback)
- [x] `chroma_search(query, k, prefer_city)` — clean public API for Leytisha
- [x] MVP rerank: vector distance + rating boost + city match (req #2: RAG + reranking ✓)
- [x] Test queries demonstrate it works

## Apr 26 follow-ups (your scope)
- **Better rerank** — try a cross-encoder reranker (e.g. `sentence-transformers/ms-marco-MiniLM-L-6-v2`) for true two-stage retrieval.
- **Filter pre-pass** — use Chroma's `where` clause for hard filters (price_range, city, cuisine_type) before vector search.
- **Speed profiling** — benchmark single-query latency; consider switching from default embeddings to OpenAI embeddings only if quality requires it.
- **Persistence sanity check** — confirm collection survives kernel restart in Colab (may need to re-mount Drive or re-run §5).

## Coordination notes
- **With Leytisha:** the `chroma_search()` signature is the integration contract. Once stable, she can drop `simple_retrieve` and call this instead.
- **With Lauren/Ryan/Leytisha:** the schema this expects is whatever `restaurants_enriched.json` contains. If schema changes upstream, update `build_embedding_text` and the `metas` dict in §5.
